In [50]:
import wikipediaapi
import polars as pl
import time
import os

from tqdm import tqdm

wiki_wiki = wikipediaapi.Wikipedia(user_agent='Silabex (opensourceatkinson@gmail.com)', language='en')


def find_checkpoint_files(checkpoint_dir):
    files = []
    for i in range(10_000):
        file = os.path.join(checkpoint_dir, f"data_{i}.jsonl")
        if os.path.exists(file):
            files.append(file)
            continue

        if len(files) > 0:
            break
    
    return files

def find_latest_checkpoint_file(checkpoint_dir):
    files = find_checkpoint_files(checkpoint_dir)

    return files[-1]
    

def find_next_checkpoint_file(checkpoint_dir, max_checkpoint_files=10):
    latest = find_latest_checkpoint_file(checkpoint_dir)
    max = int(latest.split("data_")[1].split(".jsonl")[0])
    for i in range(max-max_checkpoint_files):
        old_file = os.path.join(checkpoint_dir, f"data_{i}.jsonl")
        try:
            os.remove(old_file)
        except Exception:
            pass

    return os.path.join(checkpoint_dir, f"data_{max+1}.jsonl")

def build_wiki_dataset(checkpoint_dir, save_every=100, limit=10_000):
    checkpoint = find_latest_checkpoint_file(checkpoint_dir)
    print("loading checkpoint file:", checkpoint)
    df = pl.read_ndjson(checkpoint, schema={
        "title": pl.String,
        "content": pl.String,
        "categories": pl.List(pl.String),
        "links_to": pl.List(pl.String),
    })

    featured_page = wiki_wiki.page("Wikipedia:Featured_articles")
    links = featured_page.links.items()
    iter = tqdm(links, desc="getting wiki articles", total=limit)
    for i, (title, page) in enumerate(iter):
        if i > limit:
            break

        article_row = df.filter(pl.col("title") == title)
        if article_row.shape[0] > 0:
            continue
        
        try:
            content = {
                "title": title,
                "content": page.text,
                "categories": list(page.categories.keys()),
                "links_to": list(page.links.keys()),
            }
        except Exception as e:
            print(f"failed  to retrieve '{title}': {e}")
            continue

        df = df.vstack(pl.DataFrame([content]))

        if (i+1)%save_every== 0:
            checkpoint = find_next_checkpoint_file(checkpoint_dir)
            df.write_ndjson(checkpoint)

        time.sleep(0.5)

build_wiki_dataset("../data/wiki")

loading checkpoint file: ../data/wiki/data_30.jsonl


getting wiki articles:  63%|██████▎   | 6286/10000 [1:49:13<50:47,  1.22it/s]  

failed  to retrieve 'United States v. Washington': ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


getting wiki articles:  67%|██████▋   | 6732/10000 [1:58:07<57:20,  1.05s/it]  
